# PSAP Wrong-Routing Labels → Feature Relationship Analysis

This notebook answers two questions in order:

1. **Which calls have a defensible wrong-PSAP label?** It compares the reported latitude/longitude **and its uncertainty area** with the routed `FCC_PSAP_ID` and PSAP jurisdiction polygons.
2. **What network/signaling patterns are related to definite misroutes?** It tests numeric and categorical relationships, repeated ECGI/eNodeB/ESRK/MME patterns, safe call-level join coverage, and a chronological screening model.

The boundary check is the label generator, not the ML model. Only `DEFINITE_MISROUTE` and `CORRECT_UNAMBIGUOUS` enter the strong-label analysis. Boundary-edge and missing-data cases remain explicitly excluded.

## 0. One-time setup and paths

First run the SQL files in the `sql` folder. They write CSV files into `C:\temp\psap_route_integrity_v1\data`.

If packages are missing, uncomment the installation line once, run it, restart the kernel, and continue.

In [ ]:
# %pip install -r requirements.txt

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT_DIR = Path(r"C:\temp\psap_route_integrity_v1")

# When the notebook is opened from an extracted folder somewhere else,
# use that folder automatically if it already contains the SQL exports.
LOCAL_DIR = Path.cwd()
if (LOCAL_DIR / "data" / "psapsim_calls.csv").exists():
    ROOT_DIR = LOCAL_DIR

RUN_SCREENING_MODEL = True   # screening only; not the final production model
MAX_CALLS = None             # e.g. 20000 for a quick trial; None analyzes all calls

DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "outputs"
print("Project root:", ROOT_DIR)

## 1. Input validation

In [ ]:
required = [
    DATA_DIR / "psapsim_calls.csv",
    DATA_DIR / "psap_boundaries_chunks.csv",
]
optional = [
    DATA_DIR / "gmlc_psapsim_features.csv",
    DATA_DIR / "lsr_csr_features.csv",
    DATA_DIR / "selected_table_columns.csv",
]

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required SQL exports are missing. Run sql/00_run_all_exports.sql first:\n"
        + "\n".join(missing)
    )

display(pd.DataFrame({
    "file": [p.name for p in required + optional],
    "required": [True] * len(required) + [False] * len(optional),
    "exists": [p.exists() for p in required + optional],
    "size_mb": [round(p.stat().st_size / 1024**2, 3) if p.exists() else None for p in required + optional],
}))

## 2. Run both analysis parts

The implementation is in `psap_analysis_core.py` beside this notebook so every labeling, join, statistical test, exclusion, and model-screening rule is inspectable.

Important conservative rules:

- **Definite misroute (`1`)**: the caller's uncertainty area does not intersect the routed PSAP, while exactly one different PSAP fully covers the uncertainty area.
- **Strong correct (`0`)**: exactly the routed PSAP fully covers the uncertainty area.
- Boundary overlaps, boundary-edge uncertainty, invalid coordinates, and missing routed boundaries are not forced into either class.
- Signaling is attached only by a unique shared transaction ID or a reciprocal nearest match using `ECGI + ESRK + time`. Unsafe joins are rejected and reported.

In [ ]:
import importlib
import psap_analysis_core
importlib.reload(psap_analysis_core)
from psap_analysis_core import run_analysis

result = run_analysis(
    ROOT_DIR,
    run_screening=RUN_SCREENING_MODEL,
    max_calls=MAX_CALLS,
)

report = result["report"]
display(Markdown(
    f"**Calls:** {report['input_rows']:,}  \n"
    f"**Usable PSAP boundaries:** {report['usable_boundary_psaps']:,}  \n"
    f"**Strong correct:** {report['strong_correct']:,}  \n"
    f"**Definite misroutes:** {report['definite_misroutes']:,}  \n"
    f"**Strong-label coverage:** {report['strong_label_coverage']:.1%}"
))

## Part 1 — Boundary-label results

In [ ]:
display(result["label_summary"])

labels_file = OUTPUT_DIR / "calls_with_boundary_labels_and_features.csv"
labels = pd.read_csv(labels_file, low_memory=False)

misroutes = labels[labels["ROUTE_INTEGRITY_STATUS"] == "DEFINITE_MISROUTE"].copy()
columns = [c for c in [
    "CALL_BEGIN_TIME_UTC", "SETUP_ECGI_HEX", "USID", "ENBID", "GMLC_ESRK",
    "ROUTED_FCC_PSAP_ID", "EXPECTED_FCC_PSAP_ID", "UNCERT_METERS_NUM",
    "DISTANCE_TO_ROUTED_BOUNDARY_M", "POS_METHOD_USED"
] if c in misroutes.columns]

print("Definite misroute examples:")
display(misroutes[columns].head(30))

In [ ]:
# Highest-volume wrong routed→expected PSAP pairs.
if len(misroutes):
    pair_summary = (
        misroutes.groupby(["ROUTED_FCC_PSAP_ID", "EXPECTED_FCC_PSAP_ID"], dropna=False)
        .agg(
            MISROUTED_CALLS=("MISROUTE_LABEL", "size"),
            ECGIS=("SETUP_ECGI_HEX", "nunique"),
            DAYS=("CALL_DATE", "nunique"),
            MEDIAN_DISTANCE_M=("DISTANCE_TO_ROUTED_BOUNDARY_M", "median"),
        )
        .reset_index()
        .sort_values("MISROUTED_CALLS", ascending=False)
    )
    display(pair_summary.head(30))
else:
    print("No definite misroutes were produced. Review label_summary.csv before any ML work.")

### Label-quality interpretation

`LIKELY_MISROUTE_BOUNDARY_AMBIGUOUS` is useful for manual review but is not a positive training label. `ROUTED_PSAP_CONSISTENT_BUT_BOUNDARY_AMBIGUOUS` is also excluded as a negative. This prevents location uncertainty near a PSAP border from becoming false ground truth.

## Part 2A — Can the other datasets be safely mapped to the labels?

In [ ]:
join_quality = pd.DataFrame(report["join_quality"])
display(join_quality)

for row in report["join_quality"]:
    if row.get("coverage", 0) < 0.80:
        print(
            f"WARNING: {row.get('source')} coverage is {row.get('coverage', 0):.1%}. "
            "Do not treat unmatched/unsafe signaling as call-level evidence."
        )

## Part 2B — Relationship and pattern tests

These are univariate discovery tests, not causal proof:

- Numeric fields: medians, Mann–Whitney test, univariate AUC, Cliff's delta, point-biserial correlation, and false-discovery correction.
- Categorical fields: bias-corrected Cramer's V, chi-square significance, per-level misroute rates, and rate ratios.
- Entities: call volume, misroute count/rate, active days, and a 95% Wilson lower bound to avoid ranking tiny groups as the worst.
- `ROLE` distinguishes predictor candidates from post-routing consequences and technical join keys.

In [ ]:
print("Numeric relationships")
display(result["numeric_associations"].head(30))

print("Categorical relationships")
display(result["categorical_associations"].head(30))

print("Highest-risk supported levels")
display(result["categorical_levels"].head(50))

In [ ]:
print("Repeated network/entity patterns")
display(result["entity_patterns"].head(75))

## Part 2C — Guarded feature-sufficiency screening

This is deliberately not the final predictor. It compares class-balanced logistic regression and random forest with a chronological 70/30 split. It reports PR-AUC against the actual positive-rate baseline and lift/recall in the highest-risk 5% and 10% of calls.

Geometry labels, actual/expected PSAP, coordinates, post-routing completion/validation fields, and transaction IDs are excluded from predictor inputs. If there are too few strong labels or the chronological split lacks both classes, screening stops instead of inventing a result.

In [ ]:
screening = report["screening"]
display(Markdown("```json\n" + json.dumps(screening, indent=2) + "\n```"))

if not result["feature_importance"].empty:
    display(result["feature_importance"].head(30))

## 3. Decision and targeted next data investigation

In [ ]:
print("Recommended next actions")
for item in report["recommendations"]:
    print("-", item)

print("\nHigh-value fields present in the four selected tables but not yet exported/used")
display(result["targeted_next_columns"].head(100))

### How to choose the next model

- If definite misroutes repeat on a small number of fixed `ECGI → ESRK → routed PSAP` paths, implement a deterministic mapping-integrity alarm first.
- If many entities are involved and the chronological screening has strong PR-AUC/lift, build an **entity-hour predictor**: previous 1–24 hour signaling/KPIs → definite misroute in the next hour.
- If screening is weak, use `targeted_next_columns.csv` to investigate only missing route-decision, MME/GMLC, Diameter, SIP, and final-PSAP fields. Do not search thousands of unrelated tables.
- Do not report ML accuracy from this notebook as final model performance; the next model needs grouped rolling time validation and a frozen feature-availability cutoff.

## 4. Output files

In [ ]:
outputs = sorted(OUTPUT_DIR.glob("*"))
display(pd.DataFrame({
    "file": [p.name for p in outputs],
    "size_kb": [round(p.stat().st_size / 1024, 1) for p in outputs],
}))
print("Saved to:", OUTPUT_DIR)